# System 1: Baseline Evaluation on Gold Standard v2

**Purpose:** Run the Monolith RAG pipeline with its Optuna-tuned `best_config` against the **v2 gold standard** (77 queries, natural-language, stratified ticker/name surface forms). This is the direct counterpart to `notebooks/experiments/sys2_rag_agent/exp_baseline_evaluation.ipynb`.

Running both notebooks on the same v2 CSV with the same RAGAS judge gives a clean apples-to-apples comparison between the two architectures.

## Reference: System 1 on the ORIGINAL v1 CSV

From `configs/best_config.yaml` (Optuna, 50 trials, best_trial=27, tuned on v1 CSV):

| Metric | System 1 on v1 |
|---|---|
| Context Precision | 0.4392 |
| Context Recall | 0.2625 |
| Faithfulness | 0.9722 |
| **Composite** | **0.5580** |

The present notebook recomputes these on v2. Delta (v2 - v1) shows how much of the v1 score was an artifact of the underspecified-queries bias.

## Methodological notes

- **HPs frozen from v1 tuning** (user decision): `chunk_size=1000`, `overlap=10%`, `bm25_weight=0.5`, `pre_k=15`, `post_k=4`. No re-tuning on v2.
- **Judge model:** `gemini-2.0-flash` (same as System 2 evaluation, documented in `EVAL_DECISION_LOG.md`).
- **Preliminary dev-run.** Numbers are diagnostic; they must not be used to re-tune System 1.
- System 1 has no entity-form-awareness by design (blind retrieval); the ticker/name sub-aggregation here measures retrieval robustness against entity surface form.

In [ ]:
import json
import logging
import os
import sys
import time
from datetime import datetime
from pathlib import Path

# Resolve project root regardless of where the notebook runs from
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
for noisy in ["httpx", "urllib3", "chromadb", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

import warnings
warnings.filterwarnings("ignore")

from src.common.ingestion import ProcessedFiling
from src.systems.rag_monolith.pipeline import MonolithRAGPipeline
from src.evaluation.gold_standard_loader import load_gold_standard
from src.evaluation.ragas_evaluator import evaluate_run

print("Imports done.")

## 1. Load gold standard and filings

In [ ]:
GOLD_CSV = PROJECT_ROOT / "notebooks" / "experiments" / "sys1_rag_monolith" / "ablation_test_data_v2.csv"
gold_items = load_gold_standard(GOLD_CSV)
print(f"Loaded {len(gold_items)} gold-standard items from {GOLD_CSV.name}")

from collections import Counter
type_counts = Counter(item.query_type for item in gold_items)
for t, c in sorted(type_counts.items()):
    print(f"  {t}: {c}")

form_counts = Counter(item.entity_form for item in gold_items)
print("\nEntity-form distribution:")
for form, c in sorted(form_counts.items(), key=lambda kv: (kv[0] is None, kv[0])):
    print(f"  {form}: {c}")

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

filings = []
for meta_file in DATA_DIR.rglob("*.meta.json"):
    md_file = meta_file.with_suffix("").with_suffix(".md")
    if md_file.exists():
        filings.append(ProcessedFiling.from_files(md_file, meta_file))

print(f"Loaded {len(filings)} filings from {DATA_DIR}")
for f in filings:
    fy = f.metadata.fiscal_year_end[:4] if f.metadata.fiscal_year_end else "?"
    print(f"  {f.metadata.ticker} FY{fy}")

## 2. Build System 1 (Monolith) pipeline with Optuna best-config HPs

The Monolith inherits retrieval HPs from `configs/best_config.yaml` (chunk_size=1000, overlap=10%, bm25_weight=0.5, pre_rerank_k=15, post_rerank_k=4). Retrieval stack is shared with System 2 — only the control flow differs (retrieve-then-generate, non-agentic).

In [ ]:
print("Building MonolithRAGPipeline...")
pipeline = MonolithRAGPipeline()
pipeline.build(filings)
print("Pipeline built.")
print(f"Params: {pipeline.params}")

## 3. Run all 77 queries

Per-query error handling (a single failure does not abort the run). Intermediate results are persisted to JSON after every query so a mid-run crash does not lose progress. Expected wall-clock time for System 1: ~5-10 min (much faster than the agent — only one retrieve+generate round per query). Expected cost: well under $1.

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
intermediate_path = RESULTS_DIR / f"sys1_baseline_raw_{run_timestamp}.json"
print(f"Intermediate results will be saved to: {intermediate_path}")


def coerce_answer_to_str(answer) -> str:
    """Gemini 2.5 may return [{'type':'text','text':...}] lists. Coerce to str."""
    if isinstance(answer, str):
        return answer
    if isinstance(answer, list):
        return "\n".join(
            p.get("text", str(p)) if isinstance(p, dict) else str(p)
            for p in answer
        )
    return str(answer)


raw_results = []
total_start = time.perf_counter()

for i, item in enumerate(gold_items, 1):
    print(f"[{i:3d}/{len(gold_items)}] {item.query_type:10s} | {item.doc_refs:25s} | {item.question[:70]}")
    try:
        res = pipeline.query(item.question)
        answer_str = coerce_answer_to_str(res.answer)
        entry = {
            "id": item.id,
            "question": item.question,
            "ground_truth": item.ground_truth,
            "query_type": item.query_type,
            "doc_refs": item.doc_refs,
            "entity_form": item.entity_form,
            "answer": answer_str,
            "contexts": res.contexts,
            "num_contexts": len(res.contexts),
            "num_steps": res.metrics.num_steps,
            "latency_seconds": res.metrics.latency_seconds,
            "prompt_tokens": res.metrics.token_usage.prompt_tokens,
            "completion_tokens": res.metrics.token_usage.completion_tokens,
            "total_tokens": res.metrics.token_usage.total_tokens,
            "estimated_cost_usd": res.metrics.estimated_cost_usd,
            "error": None,
        }
        print(
            f"       -> {res.metrics.latency_seconds:.1f}s | "
            f"{len(res.contexts)} contexts | "
            f"{res.metrics.token_usage.total_tokens} tok"
        )
    except Exception as e:
        print(f"       !! ERROR: {e}")
        entry = {
            "id": item.id,
            "question": item.question,
            "ground_truth": item.ground_truth,
            "query_type": item.query_type,
            "doc_refs": item.doc_refs,
            "entity_form": item.entity_form,
            "answer": "",
            "contexts": [],
            "num_contexts": 0,
            "num_steps": 0,
            "latency_seconds": 0.0,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
            "estimated_cost_usd": 0.0,
            "error": str(e),
        }

    raw_results.append(entry)

    with open(intermediate_path, "w", encoding="utf-8") as f:
        json.dump(raw_results, f, ensure_ascii=False, indent=2)

total_elapsed = time.perf_counter() - total_start
print(f"\nAll queries done in {total_elapsed/60:.1f} minutes. Saved to {intermediate_path}")

## 4. Aggregate efficiency metrics

In [ ]:
ok_rows = [r for r in raw_results if r["error"] is None]
err_rows = [r for r in raw_results if r["error"] is not None]

n = len(ok_rows)
if n == 0:
    raise RuntimeError("All queries errored.")

avg_latency = sum(r["latency_seconds"] for r in ok_rows) / n
avg_tokens = sum(r["total_tokens"] for r in ok_rows) / n
total_tokens = sum(r["total_tokens"] for r in ok_rows)
total_cost = sum(r["estimated_cost_usd"] for r in ok_rows)

print("Efficiency metrics (System 1, best_config on v2):")
print(f"  Successful queries:    {n} / {len(gold_items)}")
print(f"  Errored queries:       {len(err_rows)}")
print(f"  Avg latency:           {avg_latency:.2f} s")
print(f"  Avg tokens per query:  {avg_tokens:.0f}")
print(f"  Total tokens:          {total_tokens:,}")
print(f"  Estimated cost (USD):  ${total_cost:.4f}")

if err_rows:
    print("\nErrors:")
    for r in err_rows:
        print(f"  id={r['id']}: {r['error']}")

## 5. RAGAS evaluation (same judge as System 2)

In [ ]:
eval_items = [
    item for item, r in zip(gold_items, raw_results) if r["error"] is None
]
eval_answers = [r["answer"] for r in raw_results if r["error"] is None]
eval_contexts = [r["contexts"] for r in raw_results if r["error"] is None]

print(f"Running RAGAS on {len(eval_items)} successful queries...")
scores = evaluate_run(
    gold_standard=eval_items,
    answers=eval_answers,
    contexts=eval_contexts,
)

print("\nRAGAS scores (System 1 on v2):")
for k, v in scores.to_dict().items():
    print(f"  {k}: {v}")

## 5b. RAGAS sub-aggregation by entity form (ticker vs name)

Does System 1 handle ticker-phrased queries differently from name-phrased ones? A large delta suggests BM25 tokenization or retrieval bias.

In [ ]:
scores_by_form: dict[str, dict] = {}

for form in ("ticker", "name"):
    sub_items = [
        item for item, r in zip(gold_items, raw_results)
        if r["error"] is None and r.get("entity_form") == form
    ]
    sub_answers = [
        r["answer"] for r in raw_results
        if r["error"] is None and r.get("entity_form") == form
    ]
    sub_contexts = [
        r["contexts"] for r in raw_results
        if r["error"] is None and r.get("entity_form") == form
    ]

    if not sub_items:
        print(f"[{form}] no successful rows - skipping.")
        continue

    print(f"\n[{form}] Running RAGAS on {len(sub_items)} queries...")
    sub_scores = evaluate_run(
        gold_standard=sub_items,
        answers=sub_answers,
        contexts=sub_contexts,
    )
    scores_by_form[form] = sub_scores.to_dict()

    print(f"[{form}] scores:")
    for k, v in scores_by_form[form].items():
        print(f"  {k}: {v}")

print("\n" + "=" * 60)
print(f"{'Metric':<22} {'ticker':>12} {'name':>12} {'Delta':>12}")
print("-" * 60)
if "ticker" in scores_by_form and "name" in scores_by_form:
    for key in ["context_precision", "context_recall", "faithfulness", "composite_score"]:
        t = scores_by_form["ticker"][key]
        n = scores_by_form["name"][key]
        print(f"{key:<22} {t:>12.4f} {n:>12.4f} {(n-t):>+12.4f}")
    print("\nInterpretation: positive delta = System 1 performs BETTER with name form.")
    print("A large absolute delta in either direction = surface-form sensitivity.")

## 6. Side-by-side: v1 reference vs v2 current

In [ ]:
SYS1_V1_REFERENCE = {
    "context_precision": 0.4392,
    "context_recall": 0.2625,
    "faithfulness": 0.9722,
    "composite_score": 0.5580,
}

sys1_v2 = scores.to_dict()

print(f"{'Metric':<22} {'v1 ref':>12} {'v2 current':>14} {'Delta':>12}")
print("-" * 62)
for key in ["context_precision", "context_recall", "faithfulness", "composite_score"]:
    v1 = SYS1_V1_REFERENCE[key]
    v2 = sys1_v2[key]
    delta = v2 - v1
    arrow = "+" if delta >= 0 else ""
    print(f"{key:<22} {v1:>12.4f} {v2:>14.4f} {arrow}{delta:>11.4f}")

print("\nNotes:")
print("- v1 scores: Optuna-tuned on ORIGINAL CSV with underspecified queries.")
print("- v2 scores: same HPs, applied to natural-language queries with explicit entity+year.")
print("- A significant v1 > v2 delta means the v1 score was inflated by the 'lucky retrieval'")
print("  effect on ambiguous queries. v2 is the methodologically cleaner baseline.")
print("- For the Sys1-vs-Sys2 architectural comparison, use v2 numbers from BOTH notebooks.")

## 7. Persist aggregated results

In [ ]:
summary = {
    "run_timestamp": run_timestamp,
    "system": "rag_monolith",
    "gold_csv": str(GOLD_CSV.relative_to(PROJECT_ROOT)),
    "num_queries": len(gold_items),
    "num_successful": len(ok_rows),
    "num_errored": len(err_rows),
    "pipeline_params": pipeline.params,
    "efficiency": {
        "avg_latency_seconds": round(avg_latency, 3),
        "avg_tokens_per_query": int(avg_tokens),
        "total_tokens": int(total_tokens),
        "estimated_total_cost_usd": round(total_cost, 4),
    },
    "ragas_scores": scores.to_dict(),
    "ragas_scores_by_entity_form": scores_by_form,
    "sys1_v1_reference": SYS1_V1_REFERENCE,
    "disclaimer": (
        "Preliminary System 1 baseline on ablation_test_data_v2.csv with HPs frozen "
        "from v1 Optuna tuning. Diagnostic only; must not be used to re-tune System 1."
    ),
}

summary_path = RESULTS_DIR / f"sys1_baseline_summary_{run_timestamp}.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Summary saved to: {summary_path}")
print(f"Raw per-query data: {intermediate_path}")